In [1]:
# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gseapy as gp
from pathlib import Path
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Paths set up")

Paths set up


In [7]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import liana
import gc
from pathlib import Path
from scipy.sparse import issparse

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

cluster_labels_2 = {
    "0": "Endothelial cells", "1": "Endothelial cells",
    "2": "CAFs", "3": "PVL", "4": "Basal epithelial",
    "5": "B cells", "6": "Cycling cells", "7": "Plasma cells",
    "8": "Cycling epithelial", "9": "CD8 T cells",
    "10": "NK cells", "11": "T cells", "12": "Naive/memory T cells",
    "13": "Luminal epithelial", "14": "Macrophages",
    "15": "Monocytes/DC", "16": "Cycling myeloid", "17": "pDC",
    "18": "Luminal epithelial", "19": "Luminal epithelial",
    "20": "Epithelial", "21": "Epithelial",
    "22": "Luminal epithelial", "23": "Luminal epithelial",
    "24": "Luminal epithelial", "25": "Luminal epithelial"
}

print("Ready")

Ready


In [2]:
# ----------------------------
# Cell 2 — Load annotated objects
# ----------------------------
adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad")
adata2 = sc.read_h5ad(PROCESSED_DIR / "GSE176078_phase2_v2_annotated.h5ad")

# Re-add cell type labels
cluster_labels_1 = {
    "0": "T cells (resting)",
    "1": "T cells (naive/memory)",
    "2": "NK/Cytotoxic T cells",
    "3": "Activated T cells",
    "4": "Macrophages",
    "5": "Monocytes/DC"
}

cluster_labels_2 = {
    "0": "Endothelial cells",
    "1": "Endothelial cells",
    "2": "CAFs",
    "3": "PVL",
    "4": "Basal epithelial",
    "5": "B cells",
    "6": "Cycling cells",
    "7": "Plasma cells",
    "8": "Cycling epithelial",
    "9": "CD8 T cells",
    "10": "NK cells",
    "11": "T cells",
    "12": "Naive/memory T cells",
    "13": "Luminal epithelial",
    "14": "Macrophages",
    "15": "Monocytes/DC",
    "16": "Cycling myeloid",
    "17": "pDC",
    "18": "Luminal epithelial",
    "19": "Luminal epithelial",
    "20": "Epithelial",
    "21": "Epithelial",
    "22": "Luminal epithelial",
    "23": "Luminal epithelial",
    "24": "Luminal epithelial",
    "25": "Luminal epithelial"
}

adata1.obs["cell_type"] = adata1.obs["leiden_0.8"].map(cluster_labels_1)
adata2.obs["cell_type"] = adata2.obs["leiden_0.6"].map(cluster_labels_2)

print(adata1)
print(adata2)
print("\nGSE114725 tissue types:", adata1.obs["tissue"].unique().tolist())
print("GSE176078 subtypes:", adata2.obs["subtype"].unique().tolist())

AnnData object with n_obs × n_vars = 44662 × 2000
    obs: 'patient', 'tissue', 'replicate', 'cluster', 'n_genes_by_counts', 'total_counts', 'doublet_score', 'predicted_doublet', 'leiden_0.2', 'leiden_0.4', 'leiden_0.6', 'leiden_0.8', 'leiden_1.0', 'celltypist', 'cell_type'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'cell_type_colors', 'celltypist_colors', 'hvg', 'leiden_0.2', 'leiden_0.2_colors', 'leiden_0.4', 'leiden_0.4_colors', 'leiden_0.6', 'leiden_0.6_colors', 'leiden_0.8', 'leiden_0.8_colors', 'leiden_1.0', 'leiden_1.0_colors', 'log1p', 'neighbors', 'patient_colors', 'pca', 'rank_genes_leiden_0.8', 'tissue_colors', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'
AnnData object with n_obs × n_vars = 91425 × 2000
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major', 

In [3]:
from scipy.sparse import issparse
import anndata as ad

def get_raw_counts_for_celltype(adata, cell_type, cell_type_col="cell_type"):
    """Extract raw normalised counts for a specific cell type."""
    
    # Get boolean mask as numpy array
    mask = (adata.obs[cell_type_col] == cell_type).values
    
    # Subset raw matrix directly
    X_raw = adata.raw.X[mask]
    
    if issparse(X_raw):
        X_raw = X_raw.toarray()
    
    # Create small AnnData for just this cell type
    adata_ct = ad.AnnData(
        X=X_raw,
        obs=adata.obs[mask].copy(),
        var=adata.raw.var.copy()
    )
    
    print(f"{cell_type}: {adata_ct.n_obs} cells x {adata_ct.n_vars} genes")
    print(f"Sample values: {X_raw[0, :5]}")
    
    return adata_ct

# Test on T cells from GSE114725
adata1_tcells_raw = get_raw_counts_for_celltype(adata1, "T cells (resting)")

T cells (resting): 11734 cells x 14800 genes
Sample values: [0. 0. 0. 0. 0.]


In [4]:
# ----------------------------
# Cell 4 — Load raw count objects
# ----------------------------
RAW_DIR = PROJECT_DIR / "Data" / "Raw"

adata1_counts = sc.read_h5ad(RAW_DIR / "GSE114725_raw.h5ad")
adata2_counts = sc.read_h5ad(RAW_DIR / "GSE176078_raw.h5ad")

# Transfer cell type labels using cell barcodes as index
adata1_counts = adata1_counts[adata1.obs_names].copy()
adata2_counts = adata2_counts[adata2.obs_names].copy()

adata1_counts.obs["cell_type"] = adata1.obs["cell_type"].values
adata1_counts.obs["tissue"] = adata1.obs["tissue"].values
adata1_counts.obs["patient"] = adata1.obs["patient"].values

adata2_counts.obs["cell_type"] = adata2.obs["cell_type"].values
adata2_counts.obs["subtype"] = adata2.obs["subtype"].values
adata2_counts.obs["orig.ident"] = adata2.obs["orig.ident"].values

# Verify raw counts
import numpy as np
sample1 = adata1_counts.X[0:5]
if issparse(sample1):
    sample1 = sample1.toarray()
print("GSE114725 sample values (should be integers):")
print(sample1[0, :10])
print("Max value:", adata1_counts.X.max())

print("\nGSE114725:", adata1_counts.n_obs, "cells x", adata1_counts.n_vars, "genes")
print("GSE176078:", adata2_counts.n_obs, "cells x", adata2_counts.n_vars, "genes")

GSE114725 sample values (should be integers):
[0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
Max value: 1236.0

GSE114725: 44662 cells x 14875 genes
GSE176078: 91425 cells x 29733 genes


In [5]:
import liana
print(liana.__version__)

1.7.3


In [4]:
# ----------------------------
# Cell 5 — LIANA cell-cell communication setup
# ----------------------------
import liana
from liana.method import cellphonedb, natmi, connectome, logfc, singlecellsignalr, rank_aggregate

# Check available methods
print("Available LIANA methods:")
print(liana.mt.__dir__())

# Check available resources (ligand-receptor databases)
from liana.resource import select_resource
resources = liana.resource.show_resources()
print("\nAvailable resources:")
print(resources)

Available LIANA methods:
['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__path__', '__file__', '__cached__', '__builtins__', 'Callable', 'np', 'DataFrame', 'V', '_pipe_utils', 'fun', 'build_prior_network', 'find_causalnet', 'estimate_metalinks', 'sc', 'cellchat', 'cellphonedb', 'connectome', 'geometric_mean', 'logfc', 'natmi', 'scseqcomm', 'singlecellsignalr', 'Method', 'MethodMeta', '_show_methods', 'AggregateClass', 'aggregate_meta', 'sp', 'MistyData', 'bivariate', 'compute_global_specificity', 'genericMistyData', 'inflow', 'lrMistyData', '_methods', 'rank_aggregate', 'show_methods', 'get_method_scores', 'process_scores']

Available resources:
['baccin2019', 'cellcall', 'cellchatdb', 'cellinker', 'cellphonedb', 'celltalkdb', 'connectomedb2020', 'consensus', 'embrace', 'guide2pharma', 'hpmr', 'icellnet', 'italk', 'kirouac2010', 'lrdb', 'mouseconsensus', 'ramilowski2015']


In [7]:
# ----------------------------
# Cell 6 — Run LIANA on GSE114725
# ----------------------------

# LIANA needs log-normalised data in adata.X
# and cell type labels in adata.obs

# Run rank_aggregate — combines multiple methods
liana.mt.rank_aggregate(
    adata1,
    groupby="cell_type",
    resource_name="consensus",
    expr_prop=0.1,  # minimum proportion of cells expressing the gene
    min_cells=10,   # minimum cells per cell type
    verbose=True,
    use_raw=False
)

print("LIANA complete for GSE114725")
print(adata1.uns["liana_res"].head(10))

Using resource `consensus`.
Using `.X`!
Converting to sparse csr matrix!
['TCONS_00029157'] contain `_`. Consider replacing those!
0.84 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 44662 samples and 122 features


C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.


Assuming that counts were `natural` log-normalized!
Running CellPhoneDB


100%|██████████| 1000/1000 [01:55<00:00,  8.65it/s]


Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt


LIANA complete for GSE114725
              source             target ligand_complex receptor_complex  \
0  Activated T cells  Activated T cells        ADCYAP1            ADRB2   
1  Activated T cells  Activated T cells        ADCYAP1            VIPR2   
2  Activated T cells  Activated T cells            ADM            ADRB2   
3  Activated T cells  Activated T cells            ADM            RAMP1   
4  Activated T cells  Activated T cells           APOE             LRP8   
5  Activated T cells  Activated T cells           BTLA            CD247   
6  Activated T cells  Activated T cells           BTLA            CD79A   
7  Activated T cells  Activated T cells             C3            C5AR2   
8  Activated T cells  Activated T cells             C3              CR2   
9  Activated T cells  Activated T cells          CCL13            ACKR1   

   lr_means  cellphone_pvals  expr_prod  scaled_weight  lr_logfc  spec_weight  \
0  0.093880            0.000   0.008643       0.145580  1.881500

In [8]:
# Cell 2 — Load GSE176078 only and subset to immune cells
adata2 = sc.read_h5ad(PROCESSED_DIR / "GSE176078_phase2_v2_annotated.h5ad")
adata2.obs["cell_type"] = adata2.obs["leiden_0.6"].map(cluster_labels_2)

immune_types = [
    "T cells", "CD8 T cells", "NK cells", "Naive/memory T cells",
    "B cells", "Plasma cells", "Macrophages", "Monocytes/DC", "pDC"
]

mask = adata2.obs["cell_type"].isin(immune_types).values
adata2_immune = adata2[mask].copy()

# Free up memory immediately
del adata2
gc.collect()

print(f"Immune cells: {adata2_immune.n_obs}")
print(adata2_immune.obs["cell_type"].value_counts())

Immune cells: 43820
cell_type
Naive/memory T cells    11590
CD8 T cells              9387
Macrophages              8705
T cells                  5989
B cells                  2791
Plasma cells             2583
NK cells                 2438
pDC                       315
Monocytes/DC               22
Name: count, dtype: int64


In [9]:
# Cell 3 — Run LIANA
liana.mt.rank_aggregate(
    adata2_immune,
    groupby="cell_type",
    resource_name="consensus",
    expr_prop=0.1,
    min_cells=10,
    verbose=True,
    use_raw=False
)

print("LIANA complete for GSE176078")
print(adata2_immune.uns["liana_res"].head(10))

Using resource `consensus`.
Using `.X`!
Converting to sparse csr matrix!
Converting `cell_type` to categorical!
0.72 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 43820 samples and 259 features


C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.


Assuming that counts were `natural` log-normalized!
Running CellPhoneDB


100%|██████████| 1000/1000 [03:22<00:00,  4.94it/s]


Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt


LIANA complete for GSE176078
    source   target ligand_complex receptor_complex  lr_means  \
0  B cells  B cells            A2M             LRP1 -0.383851   
1  B cells  B cells        ADCYAP1            RAMP2 -0.163373   
2  B cells  B cells        ADCYAP1            RAMP3 -0.135479   
3  B cells  B cells        ADCYAP1             TSHR -0.037416   
4  B cells  B cells            ADM           CALCRL -0.245171   
5  B cells  B cells            ADM          MRGPRX2 -0.134285   
6  B cells  B cells            ADM            RAMP1 -0.258576   
7  B cells  B cells            ADM            RAMP2 -0.285603   
8  B cells  B cells            ADM            RAMP3 -0.257709   
9  B cells  B cells            ADM             TSHR -0.159646   

   cellphone_pvals  expr_prod  scaled_weight  lr_logfc  spec_weight  lrscore  \
0            1.000   0.144124      -0.288523 -1.028656     0.115355      NaN   
1            0.061   0.005683       0.013034 -0.802139     0.013117      NaN   
2            0.

In [10]:
# ----------------------------
# Cell 4 — Filter and visualise LIANA results
# ----------------------------

# Get results
liana_res2 = adata2_immune.uns["liana_res"]

# Filter to significant interactions
# specificity_rank < 0.05 means the interaction is specific
# magnitude_rank not always available so use specificity_rank
sig_interactions2 = liana_res2[
    liana_res2["specificity_rank"] < 0.05
].copy()

print(f"Total interactions: {len(liana_res2)}")
print(f"Significant interactions (specificity_rank < 0.05): {len(sig_interactions2)}")

# Top interactions by specificity
top_interactions2 = sig_interactions2.nsmallest(20, "specificity_rank")
print("\nTop 20 significant interactions:")
print(top_interactions2[["source", "target", "ligand_complex", "receptor_complex", "specificity_rank"]].to_string())

# Save results
liana_res2.to_csv(RESULTS_DIR / "GSE176078_liana_results.csv", index=False)
sig_interactions2.to_csv(RESULTS_DIR / "GSE176078_liana_significant.csv", index=False)
print("\nSaved")

Total interactions: 25434
Significant interactions (specificity_rank < 0.05): 3086

Top 20 significant interactions:
             source                target ligand_complex receptor_complex  specificity_rank
24760           pDC          Plasma cells            SCT             TSHR      9.272740e-07
10110  Monocytes/DC  Naive/memory T cells          CCL19             CCR7      3.708124e-06
10422  Monocytes/DC          Plasma cells          CCL19            ACKR4      3.708124e-06
11053  Monocytes/DC                   pDC          CCL19            CXCR3      4.486594e-06
9482   Monocytes/DC          Monocytes/DC          CCL19             CCR7      5.339138e-06
9255   Monocytes/DC           Macrophages          CXCL9           FCGR2A      1.014833e-05
9167   Monocytes/DC           Macrophages          CCL19           ADRA2A      1.793603e-05
13933      NK cells                   pDC           CSF2            IL3RA      2.072218e-05
8540   Monocytes/DC               B cells          CCL1

In [11]:
# ----------------------------
# Cell 5 — Save GSE114725 LIANA results and visualise both
# ----------------------------

# Load GSE114725 results from earlier session
# Need to reload adata1 since kernel was restarted
adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad")

cluster_labels_1 = {
    "0": "T cells (resting)", "1": "T cells (naive/memory)",
    "2": "NK/Cytotoxic T cells", "3": "Activated T cells",
    "4": "Macrophages", "5": "Monocytes/DC"
}
adata1.obs["cell_type"] = adata1.obs["leiden_0.8"].map(cluster_labels_1)

# Re-run LIANA on GSE114725
liana.mt.rank_aggregate(
    adata1,
    groupby="cell_type",
    resource_name="consensus",
    expr_prop=0.1,
    min_cells=10,
    verbose=True,
    use_raw=False
)

liana_res1 = adata1.uns["liana_res"]
sig_interactions1 = liana_res1[liana_res1["specificity_rank"] < 0.05].copy()

print(f"GSE114725 total interactions: {len(liana_res1)}")
print(f"GSE114725 significant interactions: {len(sig_interactions1)}")

liana_res1.to_csv(RESULTS_DIR / "GSE114725_liana_results.csv", index=False)
sig_interactions1.to_csv(RESULTS_DIR / "GSE114725_liana_significant.csv", index=False)
print("Saved")

Using resource `consensus`.
Using `.X`!
Converting to sparse csr matrix!
['TCONS_00029157'] contain `_`. Consider replacing those!
0.84 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 44662 samples and 122 features


C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.


Assuming that counts were `natural` log-normalized!
Running CellPhoneDB


100%|██████████| 1000/1000 [02:21<00:00,  7.06it/s]


Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt


GSE114725 total interactions: 4644
GSE114725 significant interactions: 458
Saved


In [1]:
# ----------------------------
# Cell 6 — Visualise LIANA results
# ----------------------------

# GSE114725 — top interactions dot plot
top1 = sig_interactions1.nsmallest(20, "specificity_rank")

fig, ax = plt.subplots(figsize=(12, 8))
liana.pl.dotplot(
    adata=adata1,
    colour="specificity_rank",
    size="lr_means",
    source_labels=top1["source"].unique().tolist(),
    target_labels=top1["target"].unique().tolist(),
    top_n=20,
    orderby="specificity_rank",
    orderby_ascending=True,
    figure_size=(12, 8),
    ax=ax
)
plt.title("GSE114725 — Top 20 Ligand-Receptor Interactions", fontsize=12)
plt.savefig(
    FIGURE_DIR / "GSE114725_liana_dotplot.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()
print("GSE114725 plot saved")

# GSE176078 — top interactions dot plot
top2 = sig_interactions2.nsmallest(20, "specificity_rank")

fig, ax = plt.subplots(figsize=(14, 10))
liana.pl.dotplot(
    adata=adata2_immune,
    colour="specificity_rank",
    size="lr_means",
    source_labels=top2["source"].unique().tolist(),
    target_labels=top2["target"].unique().tolist(),
    top_n=20,
    orderby="specificity_rank",
    orderby_ascending=True,
    figure_size=(14, 10),
    ax=ax
)
plt.title("GSE176078 — Top 20 Ligand-Receptor Interactions", fontsize=12)
plt.savefig(
    FIGURE_DIR / "GSE176078_liana_dotplot.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()
print("GSE176078 plot saved")

NameError: name 'sig_interactions1' is not defined